In [32]:
import warnings
# Suppress all deprecation and future warnings globally
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict, load_from_disk
import sys
import os
sys.path.insert(0, os.path.abspath("/cs/usr/roeizucker/new_storage/jupyter_notebooks/Tom_Hope_Project/refactored_code"))
import numpy as np
import yaml
# from utils.dataset_utils import create_dataset, create_dataset_dict, create_tokenizer
from src.data_extractor import extract_data, save_huggingface_dataset, load_dfs

In [41]:
paths = '''/sci/archive/michall/roeizucker/downloaded_datasets/GSM5652233_Liver-Hepatocytes-Z000000R3.hg38.bigwig
/sci/archive/michall/roeizucker/downloaded_datasets/GSM5652234_Liver-Hepatocytes-Z000000T3.hg38.bigwig
/sci/archive/michall/roeizucker/downloaded_datasets/GSM5652235_Liver-Hepatocytes-Z0000043Q.hg38.bigwig
/sci/archive/michall/roeizucker/downloaded_datasets/GSM5652236_Liver-Hepatocytes-Z0000044H.hg38.bigwig
/sci/archive/michall/roeizucker/downloaded_datasets/GSM5652237_Liver-Hepatocytes-Z0000044M.hg38.bigwig
/sci/archive/michall/roeizucker/downloaded_datasets/GSM5652238_Liver-Hepatocytes-Z00000431.hg38.bigwig'''.split("\n")
compare_path = "/sci/labs/michall/roeizucker/_kol_kora_high_only_Liver-Hepatocytes_kmer_3/_kol_kora_high_only_Liver-Hepatocytes_kmer/"

dfs = load_dfs(paths,["chr1",
    "chr2",
    "chr3",
    "chr4",
    "chr5",
    # "chr22",
    # "chr21",
    # "chr20",
    # "chr19",
    # "chr18",
],"full_position",False)

tested_index = 0
for tested_index in range(len(paths)):
    sample_name = paths[tested_index].split("-")[-1].replace(".hg38.bigwig","")
    variability_path = f"/sci/labs/michall/roeizucker/huggingface_datasets_dir/huggingface_datasets_dir/_Liver-Hepatocytes_kmer/{sample_name}_per_varaint_variability_Liver-Hepatocytes_kmer_seq_5400_datasets.csv"
    df_variability = pd.read_csv(variability_path)

    tested_dataset = dfs[0]
    used_datasets = dfs[0:tested_index] + dfs[tested_index + 1:]
    avg_used = pd.concat(
        [df.set_index("full_position")["methyl_rate"] for df in used_datasets],
        axis=1
    ).mean(axis=1)
    avg_used.name = "avg_used_methyl_rate"
    compare = (
        tested_dataset.set_index("full_position")[["methyl_rate"]]
        .rename(columns={"methyl_rate": "tested_methyl_rate"})
        .join(avg_used, how="inner")
        .dropna()
    )
    # make full_position a regular column
    compare_with_var = compare.reset_index()

    # add STD per position
    compare_with_var = compare_with_var.merge(
        df_variability[["full_position", "std"]],
        on="full_position",
        how="inner"
    )

    # split STD into 5 variability groups
    compare_with_var["variability_group"] = pd.cut(
        compare_with_var["std"],
        bins=5,
        labels=["group_1_lowest", "group_2", "group_3", "group_4", "group_5_highest"],
        include_lowest=True
    )

    # print(compare_with_var)
    def calc_metrics(group):
        y_true = group["tested_methyl_rate"].astype("float64")
        y_pred = group["avg_used_methyl_rate"].astype("float64")

        mask = np.isfinite(y_true) & np.isfinite(y_pred)

        y_true = y_true[mask]
        y_pred = y_pred[mask]

        return pd.Series({
            "n": len(y_true),
            "pearson": y_true.corr(y_pred),
            "mse": ((y_true - y_pred) ** 2).mean(),
            "mae": (y_true - y_pred).abs().mean()
        })

    metrics_by_group_atlas = (
        compare_with_var
        .groupby("variability_group")
        .apply(calc_metrics)
        .reset_index()
    )

    print(metrics_by_group_atlas)
    var_df = df_variability.copy()

    # Extract variant/CpG start position from full_position
    # Example: chr1:10468-10470 -> start = 10468
    var_df["variant_start"] = (
        var_df["full_position"]
        .str.split(":").str[1]
        .str.split("-").str[0]
        .astype("int64")
    )
    var_df["window_start"] = (
        var_df["window_id"]
        .str.split(":").str[1]
        .str.split("-").str[0]
        .astype("int64")
    )

    # Extract window_start from window_id
    # Example: chr1:5400-10800 -> window_start = 5400
    # Find the 6bp token start that contains this CpG/variant
    var_df["token_genomic_position"] = (
        var_df["window_start"] +
        ((var_df["variant_start"] - var_df["window_start"]) // 6) * 6
    )

    # If multiple CpGs fall in the same 6bp token, use the highest STD
    token_variability = (
        var_df
        .groupby(["window_id", "token_genomic_position"], as_index=False)["std"]
        .max()
        .rename(columns={"std": "token_std"})
    )

    # Split token STD into 5 variability groups
    token_variability["variability_group"] = pd.cut(
        token_variability["token_std"],
        bins=5,
        labels=["group_1_lowest", "group_2", "group_3", "group_4", "group_5_highest"],
        include_lowest=True
    )


    def calc_metrics(group):
                y_true = group["label"].astype("float64")
                y_pred = group["prediction"].astype("float64")

                mask = np.isfinite(y_true) & np.isfinite(y_pred)

                y_true = y_true[mask]
                y_pred = y_pred[mask]

                return pd.Series({
                    "n": len(y_true),
                    "pearson": y_true.corr(y_pred),
                    "mse": ((y_true - y_pred) ** 2).mean(),
                    "mae": (y_true - y_pred).abs().mean()
                })
    for directory in os.listdir(compare_path):
        if not directory.startswith(sample_name) or directory.startswith(sample_name + "_pretrain"):
            continue
        for epoch_path in (os.listdir(os.path.join(compare_path,directory))):
            if "15" not in epoch_path:
                continue
            if epoch_path.startswith("checkpoint"):
                continue
            evaluated_res_path = os.path.join(compare_path,directory,epoch_path,"eval_predictions.csv.gitbackup")
            print(os.path.join(directory,epoch_path))
            evaluated_df = pd.read_csv(evaluated_res_path)
            evaluated_with_var = evaluated_df.merge(
                token_variability,
                left_on=["window_id", "genomic_position"],
                right_on=["window_id", "token_genomic_position"],
                how="inner"
            )

            # print(evaluated_with_var[["window_id", "genomic_position", "label", "prediction", "token_std", "variability_group"]])
            

            metrics_by_group = (
                evaluated_with_var
                .groupby("variability_group",observed=False)
                .apply(calc_metrics)
                .reset_index()
            )

            print(metrics_by_group)
    print("#"*30)
    # print(df_variability)

  variability_group          n   pearson       mse       mae
0    group_1_lowest  7023951.0  0.914430  0.007379  0.054329
1           group_2  1032466.0  0.689669  0.029407  0.128185
2           group_3   146403.0  0.513939  0.062179  0.205342
3           group_4    33438.0  0.371394  0.087916  0.251371
4   group_5_highest     3136.0  0.286471  0.100060  0.264734
Z000000R3_epoch-2-step-135116_retrain_kol_kora_high_only_Liver-Hepatocytes_kmer_lr_1e-06_bs_64_seq_5400_testsize_0.3/epoch-15-step-201510
  variability_group          n   pearson       mse       mae
0    group_1_lowest  2088360.0  0.802690  0.015652  0.082660
1           group_2   315525.0  0.530976  0.040528  0.146394
2           group_3    44368.0  0.326727  0.083773  0.201714
3           group_4    10372.0  0.132780  0.145424  0.282257
4   group_5_highest      944.0  0.049006  0.167053  0.312339
Z000000R3_epoch-1-step-67558_retrain_kol_kora_high_only_Liver-Hepatocytes_kmer_lr_1e-06_bs_64_seq_5400_testsize_0.3/epoch-15-step-

Z000000R3_epoch-2-step-135116_retrain_kol_kora_high_only_Liver-Hepatocytes_kmer_lr_1e-06_bs_64_seq_5400_testsize_0.3/epoch-15-step-201510
  variability_group          n   pearson       mse       mae
0    group_1_lowest  2088360.0  0.802690  0.015652  0.082660
1           group_2   315525.0  0.530976  0.040528  0.146394
2           group_3    44368.0  0.326727  0.083773  0.201714
3           group_4    10372.0  0.132780  0.145424  0.282257
4   group_5_highest      944.0  0.049006  0.167053  0.312339
Z000000R3_epoch-1-step-67558_retrain_kol_kora_high_only_Liver-Hepatocytes_kmer_lr_1e-06_bs_64_seq_5400_testsize_0.3/epoch-15-step-201510
  variability_group          n   pearson       mse       mae
0    group_1_lowest  2088360.0  0.762735  0.018511  0.091901
1           group_2   315525.0  0.497551  0.043359  0.152772
2           group_3    44368.0  0.310066  0.084125  0.204321
3           group_4    10372.0  0.122974  0.143384  0.282068
4   group_5_highest      944.0  0.047068  0.162355  0.

,window_id,window_start,window_end,token_index,token_position_in_window,base_position_in_window,genomic_position,label,prediction,logit,probability,token_genomic_position,token_std,variability_group
0,chr1:100002600-100008000,100002600,100008000,64,63,378,100002978,0.949219,0.951863,2.984375,0.951863,100002978,0.03069,group_1_lowest
1,chr1:100002600-100008000,100002600,100008000,93,92,552,100003152,1.000000,0.970688,3.500000,0.970688,100003152,0.03550,group_1_lowest
2,chr1:100002600-100008000,100002600,100008000,96,95,570,100003170,0.974121,0.964321,3.296875,0.964321,100003170,0.03076,group_1_lowest
3,chr1:100002600-100008000,100002600,100008000,224,223,1338,100003938,0.945801,0.878314,1.976562,0.878314,100003938,0.06610,group_1_lowest
4,chr1:100002600-100008000,100002600,100008000,325,324,1944,100004544,0.984863,0.972415,3.562500,0.972415,100004544,0.01952,group_1_lowest
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2459564,chr5:99997200-100002600,99997200,100002600,224,223,1338,99998538,0.278076,0.353986,-0.601562,0.353986,99998538,0.07135,group_1_lowest
2459565,chr5:99997200-100002600,99997200,100002600,365,364,2184,99999384,0.777832,0.576065,0.306641,0.576065,99999384,0.13810,group_2
2459566,chr5:99997200-100002600,99997200,100002600,384,383,2298,99999498,0.570801,0.598312,0.398438,0.598312,99999498,0.06140,group_1_lowest
2459567,chr5:99997200-100002600,99997200,100002600,853,852,5112,100002312,0.708008,0.577972,0.314453,0.577972,100002312,0.06490,group_1_lowest


Z000000R3_epoch-2-step-135116_retrain_kol_kora_high_only_Liver-Hepatocytes_kmer_lr_1e-06_bs_64_seq_5400_testsize_0.3/epoch-15-step-201510


/tmp/ipykernel_1327317/1515450905.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_metrics)


  variability_group          n   pearson       mse       mae
0    group_1_lowest  2088360.0  0.802690  0.015652  0.082660
1           group_2   315525.0  0.530976  0.040528  0.146394
2           group_3    44368.0  0.326727  0.083773  0.201714
3           group_4    10372.0  0.132780  0.145424  0.282257
4   group_5_highest      944.0  0.049006  0.167053  0.312339
Z000000R3_epoch-1-step-67558_retrain_kol_kora_high_only_Liver-Hepatocytes_kmer_lr_1e-06_bs_64_seq_5400_testsize_0.3/epoch-15-step-201510


/tmp/ipykernel_1327317/1515450905.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_metrics)


  variability_group          n   pearson       mse       mae
0    group_1_lowest  2088360.0  0.762735  0.018511  0.091901
1           group_2   315525.0  0.497551  0.043359  0.152772
2           group_3    44368.0  0.310066  0.084125  0.204321
3           group_4    10372.0  0.122974  0.143384  0.282068
4   group_5_highest      944.0  0.047068  0.162355  0.308656
Z000000R3_no_pretraining_retrain_kol_kora_high_only_Liver-Hepatocytes_kmer_lr_1e-06_bs_64_seq_5400_testsize_0.3/epoch-15-step-201510
  variability_group          n   pearson       mse       mae
0    group_1_lowest  2088360.0  0.756797  0.019091  0.090653
1           group_2   315525.0  0.570321  0.040728  0.147912
2           group_3    44368.0  0.397561  0.079956  0.193731
3           group_4    10372.0  0.173136  0.145080  0.280092
4   group_5_highest      944.0  0.140030  0.163795  0.306863


/tmp/ipykernel_1327317/1515450905.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_metrics)


In [ ]:


# evaluated_res_path = "/sci/labs/michall/roeizucker/_kol_kora_high_only_Liver-Hepatocytes_kmer_3/_kol_kora_high_only_Liver-Hepatocytes_kmer/Z000000R3_no_pretraining_retrain_kol_kora_high_only_Liver-Hepatocytes_kmer_lr_1e-06_bs_64_seq_5400_testsize_0.3/epoch-13-step-174642/eval_predictions.csv.gitbackup"
# evaluated_df = pd.read_csv(evaluated_res_path)
# evaluated_with_var = evaluated_df.merge(
#     token_variability,
#     left_on=["window_id", "genomic_position"],
#     right_on=["window_id", "token_genomic_position"],
#     how="inner"
# )

# # print(evaluated_with_var[["window_id", "genomic_position", "label", "prediction", "token_std", "variability_group"]])
# def calc_metrics(group):
#     y_true = group["label"].astype("float64")
#     y_pred = group["prediction"].astype("float64")

#     mask = np.isfinite(y_true) & np.isfinite(y_pred)

#     y_true = y_true[mask]
#     y_pred = y_pred[mask]

#     return pd.Series({
#         "n": len(y_true),
#         "pearson": y_true.corr(y_pred),
#         "mse": ((y_true - y_pred) ** 2).mean(),
#         "mae": (y_true - y_pred).abs().mean()
#     })

# metrics_by_group = (
#     evaluated_with_var
#     .groupby("variability_group")
#     .apply(calc_metrics)
#     .reset_index()
# )

# print(metrics_by_group)


In [ ]:
# import numpy as np
# import pandas as pd

# # make full_position a regular column
# compare_with_var = compare.reset_index()

# # add STD per position
# compare_with_var = compare_with_var.merge(
#     df_variability[["full_position", "std"]],
#     on="full_position",
#     how="inner"
# )

# # split STD into 5 variability groups
# compare_with_var["variability_group"] = pd.cut(
#     compare_with_var["std"],
#     bins=5,
#     labels=["group_1_lowest", "group_2", "group_3", "group_4", "group_5_highest"],
#     include_lowest=True
# )

# # print(compare_with_var)
# def calc_metrics(group):
#     y_true = group["tested_methyl_rate"].astype("float64")
#     y_pred = group["avg_used_methyl_rate"].astype("float64")

#     mask = np.isfinite(y_true) & np.isfinite(y_pred)

#     y_true = y_true[mask]
#     y_pred = y_pred[mask]

#     return pd.Series({
#         "n": len(y_true),
#         "pearson": y_true.corr(y_pred),
#         "mse": ((y_true - y_pred) ** 2).mean(),
#         "mae": (y_true - y_pred).abs().mean()
#     })

# metrics_by_group_pred = (
#     compare_with_var
#     .groupby("variability_group")
#     .apply(calc_metrics)
#     .reset_index()
# )

# # print(metrics_by_group_pred)
# std_by_position = (
#     df_variability
#     .groupby("full_position", as_index=False)["std"]
#     .max()
# )


  variability_group          n   pearson       mse       mae
0    group_1_lowest  2090765.0  0.780585  0.020640  0.093410
1           group_2   300492.0  0.549651  0.055476  0.175001
2           group_3    55497.0  0.480613  0.101630  0.224860
3           group_4    11709.0  0.450354  0.149279  0.282584
4   group_5_highest      814.0  0.125681  0.190092  0.328327
  variability_group          n   pearson       mse       mae
0    group_1_lowest  7001141.0  0.734847  0.023989  0.098480
1           group_2  1008588.0  0.506896  0.045514  0.158442
2           group_3   189494.0  0.355611  0.078501  0.223725
3           group_4    40452.0  0.160961  0.109143  0.278350
4   group_5_highest     3186.0  0.298589  0.101264  0.269926
